<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/2_2_Metric_Methods_k%E2%80%91Nearest_Neighbors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part II. Classification. Metric Methods: k‑Nearest Neighbors

# Глава 5. Метрические методы: k‑ближайших соседей

В предыдущей главе мы изучили логистическую регрессию — параметрический метод классификации, который предполагает, что логарифм шансов линейно зависит от признаков. Это мощный и интерпретируемый инструмент, но его ограничение очевидно: реальная граница между классами редко бывает строго линейной. Мы могли бы добавить полиномиальные признаки или базисные функции, но как тогда выбрать их вид? И что, если мы вообще не хотим делать никаких предположений о форме границы?
>
> **k‑ближайших соседей (k‑NN)** предлагает радикально иной подход: никаких параметров, никакой явной функции, никакого обучения в привычном смысле. Вместо этого алгоритм запоминает все обучающие данные и классифицирует новый объект, глядя на его «соседей». Это непараметрический метод, который служит естественным мостом между линейными моделями и полностью гибкими нелинейными подходами.
| Характеристика | Логистическая регрессия | k‑ближайших соседей |
|---------------|-------------------------|---------------------|
| Тип | Параметрическая | Непараметрическая |
| Обучение | Оптимизация весов (градиент) | Запоминание данных |
| Предсказание | Вычисление сигмоиды | Поиск соседей |
| Интерпретация | Коэффициенты → шансы | Нет явной интерпретации |
| Граница | Гладкая (линейная/полиномиальная) | Кусочно-постоянная |
| Чувствительность к выбросам | Низкая (регуляризация) | Высокая (особенно при малых k) |
| Проклятие размерности | Умеренное | Катастрофическое |

 В этой главе мы погрузимся в геометрию классификации, изучим компромисс смещения и дисперсии на примере k‑NN и увидим, как простой принцип «большинство соседей» приводит к удивительно глубоким результатам.






## 1. Идея метода: геометрия и формализация

Фундаментальный принцип метода ближайших соседей можно выразить максимой: **«скажи мне, кто твои соседи, и я скажу, кто ты»**. В задаче классификации новый объект получает метку, которую имеет большинство из $k$ его ближайших соседей в обучающей выборке; для регрессии откликом служит среднее (или взвешенное среднее) значений целевой переменной этих соседей. Интуитивное обоснование очевидно: объекты, расположенные в признаковом пространстве достаточно близко друг к другу, с высокой вероятностью обладают схожими свойствами.

Однако за внешней простотой скрывается строгая вероятностная логика. Пусть среди $k$ ближайших соседей точки $x$ ровно $k_c$ объектов принадлежат классу $c$. Тогда решающее правило большинства можно представить в эквивалентном виде

$$
\hat{y}(x) = \arg\max_{c} \frac{k_c}{k}.
$$

Дробь $k_c / k$ есть не что иное, как **непараметрическая оценка апостериорной вероятности**:

$$
\hat{P}(Y = c \mid X = x) = \frac{k_c}{k}. \tag{1.1}
$$

Таким образом, метод k‑NN осуществляет локальную оценку вероятностей классов и принимает решение в пользу наиболее вероятного из них. Это напрямую связывает его с байесовским решающим правилом: если оценка (1.1) состоятельна, то при неограниченном росте объёма данных классификатор k‑NN сходится к оптимальному байесовскому классификатору (подробнее см. раздел 3).

### 1.1 Формальное определение

Пусть обучающая выборка состоит из $n$ пар $\{(x_i, y_i)\}_{i=1}^n$, где $x_i \in \mathbb{R}^D$ — вектор признаков, а $y_i$ принимает значения из конечного множества $\{1, 2, \dots, C\}$ (классификация) или $y_i \in \mathbb{R}$ (регрессия). Для нового объекта $x$ определим множество индексов его $k$ ближайших соседей относительно наперёд заданной метрики $d$:

$$
N_k(x) = \{ i_{(1)}, i_{(2)}, \dots, i_{(k)} \} \subset \{1,\dots,n\},
$$

где индексы упорядочены по возрастанию расстояния:

$$
d(x, x_{i_{(1)}}) \le d(x, x_{i_{(2)}}) \le \dots \le d(x, x_{i_{(k)}}) \le d(x, x_j), \quad \forall j \notin N_k(x). \tag{1.2}
$$

**Классификация** выполняется по мажоритарному принципу:

$$
\hat{y}(x) = \arg\max_{c \in \{1,\dots,C\}} \sum_{i \in N_k(x)} \mathbb{1}(y_i = c), \tag{1.3}
$$

где $\mathbb{1}(\cdot)$ — индикаторная функция, равная 1, если условие истинно, и 0 в противном случае. При чётном $k$ возможна ситуация равенства голосов; стандартные рекомендации предписывают в таких случаях либо уменьшать $k$ на единицу, либо использовать случайный выбор среди классов, набравших одинаковое число голосов. На практике для бинарной классификации часто выбирают нечётные значения $k$.

**Регрессия** использует простое арифметическое среднее:

$$
\hat{y}(x) = \frac{1}{k} \sum_{i \in N_k(x)} y_i. \tag{1.4}
$$

Это локально-постоянная аппроксимация функции регрессии $m(x) = \mathbb{E}[Y \mid X = x]$. Действительно, вводя веса

$$
W_i(x) = \begin{cases}
1/k, & i \in N_k(x), \\
0, & \text{иначе},
\end{cases}
$$

мы получаем представление

$$
\hat{m}(x) = \sum_{i=1}^n W_i(x) \, y_i,
$$

которое является частным случаем **ядерной оценки Надарая–Ватсона** (Nadaraya, 1964; Watson, 1964) с равномерным ядром и адаптивной шириной окна, определяемой локальной плотностью данных через число соседей $k$. В отличие от методов с фиксированной шириной окна, здесь окно автоматически расширяется в областях малой плотности и сужается в плотных областях, что обеспечивает определённую устойчивость оценок.





### 1.2 Метрика и её роль

Центральным понятием метода является **расстояние** между объектами. Функция $d: \mathbb{R}^D \times \mathbb{R}^D \to \mathbb{R}_+$ называется метрикой, если для любых $x, x', x''$ выполняются аксиомы:

1. **Невырожденность**: $d(x, x') \ge 0$, причём $d(x, x') = 0 \iff x = x'$;
2. **Симметричность**: $d(x, x') = d(x', x)$;
3. **Неравенство треугольника**: $d(x, x') \le d(x, x'') + d(x'', x')$.

Соблюдение этих свойств гарантирует, что геометрическая интуиция о «близости» не противоречит формальным операциям. В практических реализациях иногда используются и меры, не удовлетворяющие всем трём аксиомам (например, косинусное расстояние), что требует осторожности при построении индексов и интерпретации окрестностей.

Наиболее общим семейством метрик в $\mathbb{R}^D$ является **расстояние Минковского** порядка $p \ge 1$:

$$
\|x - x'\|_p = \left( \sum_{j=1}^D |x^{(j)} - x'^{(j)}|^p \right)^{1/p}. \tag{1.5}
$$

Три важнейших частных случая:

- $p=1$ — **манхэттенское расстояние** (сумма модулей разностей);
- $p=2$ — **евклидово расстояние** (геометрическая длина);
- $p \to \infty$ — **равномерная норма** $\|x - x'\|_\infty = \max_j |x^{(j)} - x'^{(j)}|$.

В двумерном пространстве наглядно видна трансформация единичного шара $\{u : \|u\|_p \le 1\}$ при изменении $p$: при $p=1$ это ромб с вершинами на осях координат, при $p=2$ — круг, при $p=4$ — фигура, близкая к квадрату со скруглёнными углами, а в пределе $p \to \infty$ — квадрат со стороной $2$, параллельной координатным осям. Чем меньше $p$, тем сильнее метрика «поощряет» большие разности по отдельным координатам и может игнорировать малые расхождения по остальным; большие $p$, напротив, стремятся уравнять вклад всех измерений.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_vals = [1, 2, 4, np.inf]
labels = ['1 (ромб)', '2 (круг)', '4', '∞ (квадрат)']
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, p, lab in zip(axes, p_vals, labels):
    u = np.linspace(-1, 1, 401)
    v = np.linspace(-1, 1, 401)
    U, V = np.meshgrid(u, v)
    if np.isinf(p):
        dist = np.maximum(np.abs(U), np.abs(V))
    else:
        dist = (np.abs(U)**p + np.abs(V)**p)**(1/p)
    ax.contour(U, V, dist, levels=[1], colors='k')
    ax.set_aspect('equal')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_title(f'p = {lab}')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.grid(alpha=0.3)

plt.suptitle('Единичные шары в метрике Минковского для разных $p$')
plt.tight_layout()
plt.show()


Для **текстовых и высокоразмерных разреженных данных** часто применяют **косинусное расстояние**:

$$
d_{\cos}(x, x') = 1 - \frac{\langle x, x' \rangle}{\|x\|_2 \|x'\|_2}, \tag{1.6}
$$

измеряющее угол между векторами. Оно инвариантно к масштабу признаков и хорошо зарекомендовало себя в моделях представления текстов (bag‑of‑words, TF‑IDF). Однако косинусная мера не удовлетворяет неравенству треугольника без дополнительной нормировки, и её использование в некоторых структурах данных (например, в KD-деревьях) требует модификаций.

Когда признаки имеют иную природу, нежели непрерывные числовые переменные, на первый план выходят другие меры близости.

**Расстояние Жаккара** предназначено для объектов, представимых в виде множеств, — например, наборов ключевых слов, тегов или генов. Оно основано на коэффициенте Жаккара (отношении мощности пересечения к мощности объединения) и определяется как

$$
d_{\text{Jaccard}}(A, B) = 1 - \frac{|A \cap B|}{|A \cup B|}. \tag{1.7}
$$

Расстояние Жаккара является метрикой и не требует векторного представления объектов. Оно особенно удобно в задачах анализа текстов (сравнение наборов термов) или в биоинформатике, где объекты естественно описываются множествами признаков.

**Расстояние Хэмминга** широко применяется для бинарных векторов или категориальных переменных, закодированных в виде one‑hot векторов. Оно подсчитывает количество позиций, в которых различаются два вектора:

$$
d_{\text{Hamming}}(x, x') = \sum_{j=1}^D \mathbb{1}(x^{(j)} \neq x'^{\,(j)}). \tag{1.8}
$$

Формально расстояние Хэмминга является частным случаем манхэттенского расстояния, но на практике его выделяют отдельно из‑за наглядности и удобства при работе с дискретными данными. Оно также удовлетворяет всем аксиомам метрики.

Для гетерогенных признаков, сочетающих числовые, бинарные и категориальные компоненты, нередко применяют взвешенные комбинации описанных мер или специализированные расстояния (например, расстояние Говера), позволяющие адаптировать вклад каждого типа признаков.

**Практическая рекомендация.** В пространствах размерности $D > 10$ разница между евклидовой и манхэттенской метриками зачастую нивелируется вследствие концентрации меры (см. раздел 3). Тем не менее выбор метрики должен быть продиктован природой данных: для однородных числовых признаков, измеренных в одинаковых единицах, предпочтительна евклидова метрика; для гетерогенных величин — манхэттенское расстояние либо взвешенные варианты, в которых веса подбираются по важности признаков. При работе с множествами или бинарными признаками следует рассматривать расстояния Жаккара и Хэмминга соответственно.



### 1.3 Подробный пример классификации

Чтобы проиллюстрировать внутренний механизм метода k‑NN, рассмотрим искусственный набор из четырёх точек в $\mathbb{R}^2$, снабжённых бинарными метками (табл. 1). Обучающая выборка намеренно взята минимальной, чтобы можно было без труда проследить каждую операцию вручную.

**Таблица 1.** Обучающая выборка для классификации

| Точка | $x_1$ | $x_2$ | Класс $y$ |
|-------|-------|-------|------------|
| A     | 1     | 2     | 0          |
| B     | 2     | 3     | 0          |
| C     | 5     | 5     | 1          |
| D     | 6     | 4     | 1          |

В качестве меры близости используется евклидово расстояние  
$$
d_2(x, x') = \sqrt{\sum_{j=1}^2 (x^{(j)} - x'^{(j)})^2}.
$$

Число соседей $k$ фиксировано равным 3.

#### 1.3.1 Принцип отбора иллюстративных запросов

При оценивании качества модели на реальных данных тестовые точки поступают из того же распределения, что и обучающая выборка, и никак не отбираются вручную. В учебном примере, напротив, мы стремимся продемонстрировать, как метод принимает решения в качественно различных областях признакового пространства. Поэтому запросы не генерируются случайно, а **целенаправленно конструируются** на основе геометрии обучающих точек, чтобы высветить три типичные ситуации:

1. **Центральная область** — точка, примерно равноудалённая от кластеров обоих классов; здесь мажоритарное голосование может определяться небольшим перевесом в локальной окрестности.
2. **Периферия одного класса** — точка, лежащая вблизи границы, но всё ещё окружённая преимущественно представителями одного класса; это иллюстрирует устойчивость предсказания, пока состав ближайших соседей не меняется.
3. **Область доминирования другого класса** — точка, смещённая в сторону противоположного класса; здесь состав соседей радикально иной, что приводит к смене предсказания.

Такой методический отбор не является алгоритмическим, но полностью определяется визуальным анализом расположения точек. Он стандартен для учебной литературы по машинному обучению и позволяет раскрыть суть локального голосования без перебора бесконечного множества запросов.

Конкретные координаты выбраны следующими:

- $x^{(1)} = (3, 3)$ — центральная область;
- $x^{(2)} = (1, 4)$ — периферия класса 0;
- $x^{(3)} = (6, 2)$ — область доминирования класса 1.

Для каждого запроса ниже приведены полные вычисления.

#### 1.3.2 Запрос $x^{(1)} = (3, 3)$

Вычислим евклидово расстояние от запроса до каждой обучающей точки:

$$
\begin{aligned}
d_2(x^{(1)}, A) &= \sqrt{(3-1)^2 + (3-2)^2} = \sqrt{4+1} = \sqrt{5} \approx 2.236,\\
d_2(x^{(1)}, B) &= \sqrt{(3-2)^2 + (3-3)^2} = \sqrt{1+0} = 1,\\
d_2(x^{(1)}, C) &= \sqrt{(3-5)^2 + (3-5)^2} = \sqrt{4+4} = \sqrt{8} \approx 2.828,\\
d_2(x^{(1)}, D) &= \sqrt{(3-6)^2 + (3-4)^2} = \sqrt{9+1} = \sqrt{10} \approx 3.162.
\end{aligned}
$$

Упорядочим индексы по возрастанию расстояния:  
$B\ (1.000),\ A\ (2.236),\ C\ (2.828),\ D\ (3.162)$.

Три ближайших соседа образуют множество $N_3 = \{B, A, C\}$; их классы — $0, 0, 1$.  
Согласно правилу (1.3) имеем

$$
\hat{y}(x^{(1)}) = \arg\max_{c \in \{0,1\}} \sum_{i \in N_3} \mathbb{1}(y_i = c).
$$

Число голосов за класс 0 равно 2, за класс 1 — 1. Следовательно, предсказанный класс $\hat{y}(x^{(1)}) = 0$.

Пользуясь интерпретацией (1.1), находим непараметрические оценки апостериорных вероятностей:

$$
\hat{P}(Y=0 \mid x^{(1)}) = \frac{2}{3} \approx 0.667,\qquad
\hat{P}(Y=1 \mid x^{(1)}) = \frac{1}{3} \approx 0.333.
$$

Максимальная вероятность принадлежит классу 0, что и определяет итоговое решение.

#### 1.3.3 Запрос $x^{(2)} = (1, 4)$

$$
\begin{aligned}
d_2(x^{(2)}, A) &= \sqrt{(1-1)^2 + (4-2)^2} = \sqrt{0+4} = 2,\\
d_2(x^{(2)}, B) &= \sqrt{(1-2)^2 + (4-3)^2} = \sqrt{1+1} = \sqrt{2} \approx 1.414,\\
d_2(x^{(2)}, C) &= \sqrt{(1-5)^2 + (4-5)^2} = \sqrt{16+1} = \sqrt{17} \approx 4.123,\\
d_2(x^{(2)}, D) &= \sqrt{(1-6)^2 + (4-4)^2} = \sqrt{25+0} = 5.
\end{aligned}
$$

Упорядочение: $B\ (1.414),\ A\ (2.000),\ C\ (4.123),\ D\ (5.000)$.

$N_3 = \{B, A, C\}$ — тот же набор соседей, что и для $x^{(1)}$. Классы соседей: $0, 0, 1$.  
Оценки вероятностей: $\hat{P}(0 \mid x^{(2)}) = 2/3$, $\hat{P}(1 \mid x^{(2)}) = 1/3$.  

Предсказанный класс — 0. Несмотря на то что точка $x^{(2)}$ лежит на периферии класса 0, локальное большинство сохраняется, и решение остаётся прежним. Этот пример демонстрирует устойчивость метода в тех областях, где ближайшее окружение не меняет своего состава.

#### 1.3.4 Запрос $x^{(3)} = (6, 2)$

$$
\begin{aligned}
d_2(x^{(3)}, A) &= \sqrt{(6-1)^2 + (2-2)^2} = \sqrt{25+0} = 5,\\
d_2(x^{(3)}, B) &= \sqrt{(6-2)^2 + (2-3)^2} = \sqrt{16+1} = \sqrt{17} \approx 4.123,\\
d_2(x^{(3)}, C) &= \sqrt{(6-5)^2 + (2-5)^2} = \sqrt{1+9} = \sqrt{10} \approx 3.162,\\
d_2(x^{(3)}, D) &= \sqrt{(6-6)^2 + (2-4)^2} = \sqrt{0+4} = 2.
\end{aligned}
$$

Упорядочение: $D\ (2.000),\ C\ (3.162),\ B\ (4.123),\ A\ (5.000)$.

$N_3 = \{D, C, B\}$; классы — $1, 1, 0$.  
Число голосов: класс 0 — 1, класс 1 — 2.  
Оценки вероятностей:

$$
\hat{P}(Y=0 \mid x^{(3)}) = \frac{1}{3},\qquad
\hat{P}(Y=1 \mid x^{(3)}) = \frac{2}{3}.
$$

Максимум достигается для класса 1, поэтому $\hat{y}(x^{(3)}) = 1$. Здесь сдвиг запроса в правую нижнюю область привёл к тому, что два из трёх ближайших соседей принадлежат классу 1, и решение изменилось.

#### 1.3.5 Анализ и выводы

Три запроса наглядно демонстрируют ключевое свойство k‑NN — **локальную адаптивность**. В левой верхней и центральной частях признакового пространства, где среди трёх ближайших соседей доминирует класс 0, классификатор устойчиво выдаёт 0. В правой нижней области, где ближайшие точки относятся преимущественно к классу 1, предсказание становится 1. Граница между классами, порождаемая таким правилом, не задаётся явной формулой, а определяется исключительно расположением обучающих точек и значением $k$. Она является кусочно-постоянной и может быть весьма изрезанной, особенно при малых $k$.

Эти примеры также иллюстрируют вероятностную природу метода: по существу, k‑NN выполняет локальную максимизацию оценки апостериорной вероятности, что напрямую связывает его с байесовским решающим правилом. В разделе 3 будет показано, что при выполнении условий $k \to \infty$ и $k/n \to 0$ оценка (1.1) состоятельна и классификатор стремится к оптимальному байесовскому.

#### 1.3.6 Визуализация

Для графической иллюстрации трёх рассмотренных запросов можно использовать следующий код на Python (требуются библиотеки `numpy` и `matplotlib`). Он строит обучающие точки, раскрашенные по классам, отмечает тестовые запросы и соединяет каждый запрос с его $k=3$ ближайшими соседями.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Обучающая выборка
X_train = np.array([[1, 2],   # A
                    [2, 3],   # B
                    [5, 5],   # C
                    [6, 4]])  # D
y_train = np.array([0, 0, 1, 1])
labels = ['A', 'B', 'C', 'D']

# Тестовые запросы
x_queries = np.array([[3, 3], [1, 4], [6, 2]])
query_names = ['x^(1)', 'x^(2)', 'x^(3)']
k = 3

def knn_predict(x):
    dist = np.linalg.norm(X_train - x, axis=1)
    idx = np.argsort(dist)[:k]
    pred = np.bincount(y_train[idx]).argmax()
    return idx, pred

fig, ax = plt.subplots(figsize=(8, 6))

# Обучающие точки
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='bwr',
           edgecolors='k', s=120, zorder=5)
for i, lab in enumerate(labels):
    ax.text(X_train[i, 0] + 0.1, X_train[i, 1] + 0.1, lab, fontsize=10)

# Запросы и соединения с соседями
colors_q = ['lime', 'orange', 'magenta']
for xq, name, col in zip(x_queries, query_names, colors_q):
    idx, pred = knn_predict(xq)
    ax.scatter(xq[0], xq[1], c=col, marker='*', s=200,
               edgecolors='k', zorder=6)
    ax.text(xq[0] + 0.15, xq[1] + 0.15, f'{name}→кл.{pred}',
            color=col, fontweight='bold')
    for i in idx:
        ax.plot([xq[0], X_train[i, 0]], [xq[1], X_train[i, 1]],
                color=col, linestyle='--', alpha=0.5, linewidth=1)

ax.set_xlim(0, 7)
ax.set_ylim(1, 6)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title(f'Классификация k‑NN (k = {k}), три запроса')
ax.legend(['Обучающие точки'] + [f'{n}' for n in query_names])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



**Описание графика.** Обучающие точки класса 0 показаны синим, класса 1 — красным. Три тестовые точки отмечены звёздами: зелёная ($x^{(1)}$), оранжевая ($x^{(2)}$) и малиновая ($x^{(3)}$). Цвет звезды соответствует предсказанному классу. Пунктирные линии соединяют каждый запрос с его тремя ближайшими соседями, позволяя непосредственно увидеть, какие точки повлияли на решение. График подтверждает аналитические расчёты и наглядно демонстрирует локальный характер метода.



### 1.4 Подробный пример регрессии

Рассмотрим задачу предсказания заработной платы по характеристикам работника. Пусть обучающая выборка состоит из пяти наблюдений, каждое из которых описывается двумя числовыми признаками:

- $x_1$ — стаж работы (число лет);
- $x_2$ — уровень образования (число лет обучения после школы);

и целевой переменной $y$ — месячная заработная плата в тысячах рублей.

Данные приведены в таблице 2.

**Таблица 2.** Обучающая выборка для регрессии

| Работник | Стаж $x_1$ (лет) | Образование $x_2$ (лет) | Зарплата $y$ (тыс. руб.) |
|----------|-------------------|--------------------------|---------------------------|
| A        | 1                 | 1                        | 30                        |
| B        | 2                 | 3                        | 45                        |
| C        | 4                 | 2                        | 55                        |
| D        | 5                 | 5                        | 80                        |
| E        | 3                 | 4                        | 60                        |

Будем использовать евклидово расстояние

$$
d_2(x, x') = \sqrt{(x_1 - x'_1)^2 + (x_2 - x'_2)^2},
$$

и зафиксируем число соседей $k = 3$. Прогноз для новой точки вычисляется по формуле (1.4) как среднее арифметическое откликов трёх ближайших соседей.

Для иллюстрации работы метода возьмём три различных запроса — гипотетических кандидатов:

- $x^{(1)} = (2, 2)$ — начинающий специалист с небольшим стажем и средним образованием;
- $x^{(2)} = (4, 4)$ — опытный сотрудник с высшим образованием;
- $x^{(3)} = (1, 4)$ — работник с минимальным стажем, но хорошим образованием.

Такой выбор, как и ранее, мотивирован желанием продемонстрировать, как предсказание адаптируется к разным областям признакового пространства.

#### 1.4.1 Запрос $x^{(1)} = (2, 2)$

Вычислим евклидовы расстояния от $x^{(1)}$ до каждого из пяти работников:

$$
\begin{aligned}
d_2(x^{(1)}, A) &= \sqrt{(2-1)^2 + (2-1)^2} = \sqrt{1+1} = \sqrt{2} \approx 1.414,\\
d_2(x^{(1)}, B) &= \sqrt{(2-2)^2 + (2-3)^2} = \sqrt{0+1} = 1,\\
d_2(x^{(1)}, C) &= \sqrt{(2-4)^2 + (2-2)^2} = \sqrt{4+0} = 2,\\
d_2(x^{(1)}, D) &= \sqrt{(2-5)^2 + (2-5)^2} = \sqrt{9+9} = \sqrt{18} \approx 4.243,\\
d_2(x^{(1)}, E) &= \sqrt{(2-3)^2 + (2-4)^2} = \sqrt{1+4} = \sqrt{5} \approx 2.236.
\end{aligned}
$$

Упорядочим по возрастанию расстояния: $B$ (1.000), $A$ (1.414), $C$ (2.000), $E$ (2.236), $D$ (4.243).  
Три ближайших соседа — $B, A, C$. Их зарплаты: $y_B = 45$, $y_A = 30$, $y_C = 55$.  
Среднее:

$$
\hat{y}(x^{(1)}) = \frac{45 + 30 + 55}{3} = \frac{130}{3} \approx 43.33 \text{ тыс. руб.}
$$

#### 1.4.2 Запрос $x^{(2)} = (4, 4)$

Расстояния:

$$
\begin{aligned}
d_2(x^{(2)}, A) &= \sqrt{(4-1)^2 + (4-1)^2} = \sqrt{9+9} = \sqrt{18} \approx 4.243,\\
d_2(x^{(2)}, B) &= \sqrt{(4-2)^2 + (4-3)^2} = \sqrt{4+1} = \sqrt{5} \approx 2.236,\\
d_2(x^{(2)}, C) &= \sqrt{(4-4)^2 + (4-2)^2} = \sqrt{0+4} = 2,\\
d_2(x^{(2)}, D) &= \sqrt{(4-5)^2 + (4-5)^2} = \sqrt{1+1} = \sqrt{2} \approx 1.414,\\
d_2(x^{(2)}, E) &= \sqrt{(4-3)^2 + (4-4)^2} = \sqrt{1+0} = 1.
\end{aligned}
$$

Упорядочение: $E$ (1.000), $D$ (1.414), $C$ (2.000), $B$ (2.236), $A$ (4.243).  
Три ближайших соседа: $E, D, C$. Их зарплаты: $y_E = 60$, $y_D = 80$, $y_C = 55$.  
Среднее:

$$
\hat{y}(x^{(2)}) = \frac{60 + 80 + 55}{3} = \frac{195}{3} = 65.0 \text{ тыс. руб.}
$$

#### 1.4.3 Запрос $x^{(3)} = (1, 4)$

Расстояния:

$$
\begin{aligned}
d_2(x^{(3)}, A) &= \sqrt{(1-1)^2 + (4-1)^2} = \sqrt{0+9} = 3,\\
d_2(x^{(3)}, B) &= \sqrt{(1-2)^2 + (4-3)^2} = \sqrt{1+1} = \sqrt{2} \approx 1.414,\\
d_2(x^{(3)}, C) &= \sqrt{(1-4)^2 + (4-2)^2} = \sqrt{9+4} = \sqrt{13} \approx 3.606,\\
d_2(x^{(3)}, D) &= \sqrt{(1-5)^2 + (4-5)^2} = \sqrt{16+1} = \sqrt{17} \approx 4.123,\\
d_2(x^{(3)}, E) &= \sqrt{(1-3)^2 + (4-4)^2} = \sqrt{4+0} = 2.
\end{aligned}
$$

Упорядочение: $B$ (1.414), $E$ (2.000), $A$ (3.000), $C$ (3.606), $D$ (4.123).  
Три ближайших соседа: $B, E, A$. Зарплаты: $y_B = 45$, $y_E = 60$, $y_A = 30$.  
Среднее:

$$
\hat{y}(x^{(3)}) = \frac{45 + 60 + 30}{3} = \frac{135}{3} = 45.0 \text{ тыс. руб.}
$$

#### 1.4.4 Обсуждение результатов

Полученные прогнозы хорошо согласуются с интуицией:

- Начинающий специалист со стажем 2 года и образованием 2 года оценивается в 43.3 тыс. руб.
- Опытный работник со стажем 4 года и образованием 4 года — в 65 тыс. руб.
- Кандидат с минимальным стажем, но высоким образованием (4 года обучения) — в 45 тыс. руб., что выше, чем у первого запроса, благодаря присутствию в окрестности более высокооплачиваемого соседа $E$.

Метод k‑NN улавливает локальные закономерности: в центральной и левой областях зарплаты ниже, в правой верхней — выше.

#### 1.4.5 Влияние числа соседей $k$

Проанализируем точку $x^{(2)}=(4,4)$. Последовательность соседей по удалённости: $E, D, C, B, A$. Проследим изменение прогноза:

- $k=1$: только $E$, $\hat{y} = 60$;
- $k=2$: $E$ и $D$, $\hat{y} = (60+80)/2 = 70$;
- $k=3$: $E, D, C$, $\hat{y} = 65$;
- $k=4$: добавляется $B$, $\hat{y} = (60+80+55+45)/4 = 60$;
- $k=5$: все точки, $\hat{y} = (30+45+55+80+60)/5 = 54$.

При $k=1$ прогноз максимально чувствителен к единственному ближайшему соседу ($E$ с зарплатой 60 тыс. руб.). При $k=2$ резко возрастает до 70 за счёт высокооплачиваемого $D$. Далее добавление более далёких точек с меньшими зарплатами снижает оценку вплоть до глобального среднего 54. Этот эффект — прямое проявление компромисса между смещением и дисперсией, подробно рассматриваемого в разделе 2.

#### 1.4.6 Визуализация

Код на Python (требуются `numpy` и `matplotlib`) строит график с обучающими точками, тестовыми запросами и соединениями с тремя ближайшими соседями каждого запроса.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Обучающая выборка (стаж, образование, зарплата)
X_train = np.array([[1, 1],   # A
                    [2, 3],   # B
                    [4, 2],   # C
                    [5, 5],   # D
                    [3, 4]])  # E
y_train = np.array([30.0, 45.0, 55.0, 80.0, 60.0])
labels = ['A', 'B', 'C', 'D', 'E']

# Три тестовых запроса
x_queries = np.array([[2, 2], [4, 4], [1, 4]])
query_names = ['x^(1)', 'x^(2)', 'x^(3)']
k = 3

def knn_predict_reg(x):
    dist = np.linalg.norm(X_train - x, axis=1)
    idx = np.argsort(dist)[:k]
    pred = np.mean(y_train[idx])
    return idx, pred

fig, ax = plt.subplots(figsize=(8, 6))

# Обучающие точки
sc = ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='viridis',
                edgecolors='k', s=150, zorder=5)
plt.colorbar(sc, ax=ax, label='Зарплата, тыс. руб.')
for i, lab in enumerate(labels):
    ax.text(X_train[i, 0] + 0.1, X_train[i, 1] + 0.1, lab, fontsize=10)

# Тестовые запросы и соединения с соседями
colors_q = ['lime', 'orange', 'magenta']
for xq, name, col in zip(x_queries, query_names, colors_q):
    idx, pred = knn_predict_reg(xq)
    ax.scatter(xq[0], xq[1], c=col, marker='*', s=250,
               edgecolors='k', zorder=6)
    ax.text(xq[0] + 0.15, xq[1] + 0.15, f'{name}→{pred:.1f} тыс.',
            color=col, fontweight='bold')
    for i in idx:
        ax.plot([xq[0], X_train[i, 0]], [xq[1], X_train[i, 1]],
                color=col, linestyle='--', alpha=0.5, linewidth=1)

ax.set_xlim(0, 6)
ax.set_ylim(0, 6)
ax.set_xlabel('Стаж (лет)')
ax.set_ylabel('Образование (лет)')
ax.set_title(f'Регрессия k‑NN (k = {k}), три запроса')
ax.legend(['Обучающие точки'] + [f'{n}' for n in query_names])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Интерпретация графика.** Цвет точек отражает уровень зарплаты: от фиолетового (низкая) до жёлтого (высокая). Три звезды соответствуют запросам; рядом указан прогноз в тыс. рублей. Пунктирные линии показывают трёх соседей, по которым выполнялось усреднение. Видно, что в центре (стаж 2, образование 2) зарплата оценивается примерно в 43.3 тыс., в правом верхнем углу (стаж 4, образование 4) — в 65 тыс., а при малом стаже, но хорошем образовании (1;4) — в 45 тыс. График наглядно демонстрирует локальную адаптивность метода.

Таким образом, регрессия k‑NN, как и классификация, опирается на поиск ближайших соседей, но агрегирует их отклики усреднением. Выбор $k$ и метрики напрямую влияет на гладкость и точность предсказаний. Этот простой пример с зарплатами показывает, как метод может применяться для оценки непрерывных величин на практике.



## 2. Выбор гиперпараметров

Метод k‑NN характеризуется небольшим числом гиперпараметров, однако их правильный выбор критичен для достижения высокого качества предсказаний. Три основных элемента, требующих настройки: **число соседей $k$** (или ширина окна $h$ в ядерной формулировке), **метрика расстояния** и **весовая схема** агрегирования ответов соседей.

### 2.1 Число соседей и компромисс смещения и дисперсии

Центральным гиперпараметром является **число ближайших соседей $k$**. Его влияние удобно анализировать в терминах фундаментального компромисса между смещением и дисперсией (bias‑variance tradeoff), который для произвольной модели $\hat{f}$ в задаче регрессии выражается разложением среднеквадратичной ошибки в точке $x$:

$$
\mathbb{E}\bigl[ (Y - \hat{f}(x))^2 \mid X = x \bigr] = \bigl( \text{Bias}[\hat{f}(x)] \bigr)^2 + \text{Var}[\hat{f}(x)] + \sigma^2, \tag{2.1}
$$

где $\sigma^2 = \text{Var}[Y \mid X=x]$ — неустранимая ошибка (шум целевой переменной), а математические ожидания берутся по случайной обучающей выборке при фиксированном $x$.

Для k‑NN регрессии с равномерными весами предсказание имеет вид $\hat{f}(x) = \frac{1}{k} \sum_{i \in N_k(x)} y_i$. Если предположить, что в достаточно малой окрестности точки $x$ истинная функция регрессии $m(x) = \mathbb{E}[Y \mid X=x]$ меняется слабо, а отклики $y_i$ некоррелированы при фиксированных $x_i$ и имеют постоянную дисперсию $\sigma^2$, то компоненты разложения (2.1) принимают вид:

- **Смещение** определяется тем, насколько локальное среднее $m(x_i)$ по $k$ соседям отклоняется от $m(x)$. С ростом $k$ в окрестность попадают всё более удалённые точки, и смещение монотонно возрастает. Приближённо:

  $$
  \text{Bias}[\hat{f}(x)] \approx \frac{1}{k} \sum_{i \in N_k(x)} \bigl( m(x_i) - m(x) \bigr).
  $$

- **Дисперсия** предсказания при независимых и одинаково распределённых ошибках наблюдений составляет

  $$
  \text{Var}[\hat{f}(x)] = \text{Var}\!\left[ \frac{1}{k} \sum_{i=1}^k y_{(i)} \right] = \frac{\sigma^2}{k},
  $$

  то есть убывает обратно пропорционально $k$.

- **Неустранимая ошибка** $\sigma^2$ не зависит от модели и является нижней границей достижимой точности.

Таким образом, при $k=1$ смещение минимально (предсказание опирается на единственную ближайшую точку), но дисперсия максимальна и равна $\sigma^2$ — модель чрезвычайно чувствительна к случайным флуктуациям обучающих данных, что приводит к переобучению. В противоположном пределе $k=n$ предсказание сводится к глобальному среднему, дисперсия стремится к нулю, но смещение достигает максимума — модель недообучается и игнорирует всю локальную структуру данных.

Оптимальное $k$ находят как точку компромисса между этими двумя эффектами. На практике его определяют путём кросс‑валидации: разбивают обучающую выборку на $F$ блоков (фолдов), для каждого $k$ из заданной сетки вычисляют среднюю ошибку на валидационных блоках и выбирают значение, минимизирующее эту ошибку.

**Замечание о многоклассовой классификации.** Для бинарной классификации рекомендуется выбирать нечётные значения $k$ во избежание равенства голосов. Для $C$ классов при чётном $k$ возможны ничьи, и стандартные стратегии их разрешения включают: уменьшение $k$ на единицу (переход к ближайшему нечётному значению), взвешенное голосование с весами, обратно пропорциональными расстоянию (что делает ничью практически невозможной), либо случайный выбор среди классов с одинаковым числом голосов.

### 2.2 Метрика расстояния

Вторым гиперпараметром является **функция расстояния** $d(\cdot, \cdot)$, выбор которой обсуждался в разделе 1.2. Следует подчеркнуть, что в высокоразмерных пространствах ($D \gtrsim 10$) различие между многими метриками размывается из‑за эффекта концентрации меры (см. раздел 3). Тем не менее, осмысленный выбор метрики, согласованный с природой данных, остаётся важным: для однородных числовых признаков предпочтительна евклидова метрика; для гетерогенных — манхэттенское расстояние либо расстояние Говера; для разреженных текстовых данных — косинусное; для множеств — расстояние Жаккара. Метрику также можно рассматривать как гиперпараметр и выбирать по кросс‑валидации наряду с $k$.

### 2.3 Взвешенные соседи и ядерная формулировка

Третьим инструментом управления гибкостью модели является **весовая схема**. Простейший способ — назначить веса, обратно пропорциональные расстоянию до точки:

$$
\hat{y}(x) = \frac{\sum_{i \in N_k(x)} w_i y_i}{\sum_{i \in N_k(x)} w_i}, \qquad w_i = \frac{1}{d(x, x_i) + \varepsilon},
$$

где $\varepsilon > 0$ — малая константа, предотвращающая деление на ноль. Такое взвешивание делает предсказание более гладким и естественно разрешает проблему равенства голосов, поскольку даже при одинаковом числе представителей разных классов в окрестности суммарные веса будут различаться.

Более систематический подход даёт **ядерная формулировка**, в которой дискретное отсечение по числу соседей заменяется гладкой функцией окна — ядром $K: \mathbb{R}_+ \to \mathbb{R}_+$, удовлетворяющим условиям монотонного невозрастания и неотрицательности. Для классификации решающее правило принимает вид

$$
\hat{y}(x) = \arg\max_{c} \sum_{i=1}^n K\!\left( \frac{d(x, x_i)}{h} \right) \mathbb{1}(y_i = c),
$$

а для регрессии —

$$
\hat{y}(x) = \frac{ \sum_{i=1}^n K\!\left( \frac{d(x, x_i)}{h} \right) y_i }{ \sum_{i=1}^n K\!\left( \frac{d(x, x_i)}{h} \right) }. \tag{2.2}
$$

Формула (2.2) известна как **оценка Надарая–Ватсона** (Nadaraya, 1964; Watson, 1964). Параметр $h > 0$ называется **шириной окна** и играет роль, аналогичную $k$: малое $h$ даёт очень локальную, изрезанную оценку (низкое смещение, высокая дисперсия), большое $h$ размывает структуру данных (высокое смещение, низкая дисперсия). Классический k‑NN с равномерным усреднением по $k$ соседям соответствует прямоугольному ядру $K(u) = \frac{1}{2}\mathbf{1}(|u| \le 1)$ и адаптивной ширине окна $h = d(x, x_{(k)})$, определяемой по локальной плотности данных.

На практике часто используют следующие ядра (в порядке увеличения гладкости):

- **Прямоугольное**: $K(u) = \frac{1}{2}\,\mathbf{1}(|u| \le 1)$;
- **Треугольное**: $K(u) = (1 - |u|)\,\mathbf{1}(|u| \le 1)$;
- **Епанечникова**: $K(u) = \frac{3}{4}(1 - u^2)\,\mathbf{1}(|u| \le 1)$;
- **Гауссовское**: $K(u) = \frac{1}{\sqrt{2\pi}} e^{-u^2/2}$.

Вид ядра, как правило, слабо влияет на итоговое качество предсказаний, тогда как выбор $h$ (или $k$) критичен. На практике оптимальные значения $k$ и $h$ подбирают по сетке с помощью кросс‑валидации, оценивая среднюю ошибку на отложенных подвыборках.


Таким образом, настройка метода k‑NN сводится к поиску баланса трёх составляющих: числа соседей (или ширины окна), метрики расстояния и весовой схемы. Грамотный выбор этих параметров позволяет методу гибко адаптироваться к данным, не навязывая жёстких предположений об их структуре, и достигать качества, сопоставимого с существенно более сложными моделями. На практике все три гиперпараметра часто оптимизируют совместно в рамках процедуры кросс‑валидации.





## 3. Проклятие размерности

Интуитивно проклятие размерности проявляется в том, что с увеличением размерности $D$ объём пространства растёт экспоненциально. Точки, равномерно распределённые в единичном гиперкубе, оказываются в среднем всё дальше друг от друга. Формально, если $X$ и $X'$ независимы и равномерно распределены в $[0,1]^D$, то

$$
\mathbb{E}\|X - X'\|_2^2 = \frac{D}{3},
$$

откуда по неравенству Йенсена среднее евклидово расстояние асимптотически составляет $\sqrt{D/6}$ при $D \to \infty$ (строгое доказательство вынесено в задания повышенной сложности). Если в одномерном пространстве для надёжной локальной оценки требуется порядка $n$ точек, то в $D$-мерном пространстве для сохранения той же средней плотности покрытия необходимо порядка $n^D$ наблюдений. Уже при $D=10$ и скромном $n=100$ это даёт астрономическую величину, делающую точное покрытие практически невозможным.

Для метода k‑NN это означает, что понятие «близости» размывается. Более того, в высоких размерностях проявляется **эффект концентрации меры**: распределение попарных расстояний между точками сосредоточивается вокруг своего среднего, а относительная вариация расстояний стремится к нулю как $O(1/\sqrt{D})$. Иными словами, все точки выборки находятся практически на одном и том же расстоянии от любого запроса — понятие «ближайшего» соседа теряет смысл. Окрестность, содержащая $k$ ближайших соседей, становится огромной по сравнению с масштабом изменения целевой переменной, локальная аппроксимация перестаёт работать, и качество модели резко падает.

На практике k‑NN обычно приемлемо работает при $D \lesssim 20$, но для более высоких размерностей требуется либо предварительное снижение размерности (например, метод главных компонент), либо использование альтернативных методов, менее чувствительных к размерности (линейные модели, ансамбли деревьев и др.).

### 3.1 Визуализация эффекта

Приведённый ниже код на Python моделирует $n = 1000$ точек, равномерно распределённых в единичном гиперкубе, для размерностей $D = 1, 2, 5, 10, 20, 50$ и строит гистограммы попарных евклидовых расстояний. Для каждой размерности вычисляется выборочное среднее расстояние и теоретическое асимптотическое значение $\sqrt{D/6}$.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_samples = 500
dims = [1, 2, 5, 10, 20, 50]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

for ax, D in zip(axes, dims):
    X = np.random.uniform(0, 1, (n_samples, D))
    dist = np.linalg.norm(
        X[:, np.newaxis, :] - X[np.newaxis, :, :], axis=2
    )
    triu_idx = np.triu_indices(n_samples, k=1)
    all_dist = dist[triu_idx]
    mean_dist = np.mean(all_dist)
    theo_mean = np.sqrt(D / 6)

    ax.hist(all_dist, bins=50, density=True, alpha=0.7,
            color='steelblue', edgecolor='black')
    ax.axvline(mean_dist, color='red', linestyle='--', linewidth=2,
               label=f'среднее = {mean_dist:.2f}')
    ax.axvline(theo_mean, color='green', linestyle=':', linewidth=2,
               label=f'√(D/6) = {theo_mean:.2f}')
    ax.set_title(f'D = {D}')
    ax.set_xlabel('Евклидово расстояние')
    ax.set_ylabel('Плотность')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Распределение попарных расстояний для разных размерностей',
             fontsize=14)
plt.tight_layout()
plt.show()



**Интерпретация графика.** При $D=1$ и $D=2$ распределение расстояний широкое, присутствуют как очень близкие, так и далёкие пары точек — понятие «соседства» осмысленно. С ростом $D$ распределение всё сильнее концентрируется вокруг среднего, и при $D=50$ практически все пары находятся на расстоянии около $2.9$, то есть разница между «ближайшим» и «дальним» соседом становится ничтожной. Именно это явление и называют проклятием размерности для метрических методов.

**Углублённый взгляд: асимптотическая состоятельность k‑NN при $n \to \infty$.**  
Классический результат (Stone, 1977) утверждает, что если $k \to \infty$ и $k/n \to 0$ при $n \to \infty$, то оценка регрессии методом k‑NN является состоятельной оценкой условного среднего $\mathbb{E}[Y \mid X=x]$ для почти всех $x$ относительно распределения $X$. Аналогичное утверждение верно для классификации: риск k‑NN сходится к байесовскому риску. Однако скорость сходимости катастрофически зависит от размерности: для гладких функций регрессии она составляет $O(n^{-2/(2+D)})$. При $D=10$ это означает, что для уменьшения ошибки вдвое требуется увеличить объём выборки примерно в $2^{6} = 64$ раза; при $D=100$ — более чем в $10^{12}$ раз. Это делает метод крайне требовательным к объёму данных при больших $D$ и объясняет, почему на практике k‑NN редко применяют в высокоразмерных пространствах без предварительного отбора или преобразования признаков.





## 4. Алгоритмические аспекты

Наивная реализация метода k‑ближайших соседей предполагает, что для каждого тестового запроса вычисляются расстояния до всех $n$ точек обучающей выборки. Трудоёмкость такой процедуры составляет $O(Dn)$ на один запрос, где $D$ — размерность признакового пространства. Если тестовая выборка содержит $m$ объектов, суммарная сложность достигает $O(Dnm)$, что для типичных значений $n \sim 10^5$, $m \sim 10^4$, $D \sim 100$ даёт порядка $10^{11}$ операций. Это делает прямой перебор неприемлемым в большинстве практических сценариев. Более того, поскольку фаза обучения в k‑NN отсутствует, а предсказание должно выполняться быстро (часто в режиме реального времени), возникает необходимость в эффективных структурах данных для поиска ближайших соседей. Все такие структуры можно разделить на два класса: точные, гарантирующие нахождение строго ближайших точек, и приближённые, находящие достаточно близкие объекты за сублинейное время ценой небольшой потери точности.

### 4.1 Точные методы: KD‑дерево

**KD‑дерево** (k‑dimensional tree, также обозначается как $k$‑d tree) было предложено Бентли (Bentley, 1975) и представляет собой бинарное дерево, каждый узел которого соответствует разбиению пространства гиперплоскостью, параллельной одной из координатных осей. Такая структура является многомерным обобщением бинарного дерева поиска и позволяет сводить поиск ближайших соседей к обходу лишь небольшого числа узлов, избегая полного перебора.

#### 4.1.1 Построение KD‑дерева

Формально процесс построения описывается следующим рекурсивным алгоритмом. Пусть задано множество точек $\mathcal{X} = \{x_1, \dots, x_n\} \subset \mathbb{R}^D$ и текущая глубина рекурсии $d = 0$.

1. Если $\mathcal{X} = \varnothing$, возвращается пустой узел (терминальный случай для пустого поддерева).
2. Если $|\mathcal{X}| = 1$, создаётся листовой узел, содержащий единственную точку.
3. В противном случае выбирается координатная ось, по которой будет производиться разбиение: $j = d \bmod D$ (циклический перебор осей). Возможны и более сложные эвристики выбора оси — например, ось с наибольшей дисперсией значений.
4. По $j$-й координате вычисляется медиана $m = \text{median}\{x_i^{(j)} : x_i \in \mathcal{X}\}$. Точка, соответствующая медиане (или одна из них), помещается в текущий узел.
5. Множество $\mathcal{X}$ разбивается на два подмножества:
   - левое подмножество $\mathcal{X}_L = \{x \in \mathcal{X} : x^{(j)} \le m\}$;
   - правое подмножество $\mathcal{X}_R = \{x \in \mathcal{X} : x^{(j)} > m\}$.
6. Для $\mathcal{X}_L$ рекурсивно строится левое поддерево с глубиной $d+1$, для $\mathcal{X}_R$ — правое поддерево.

Узел дерева в итоге хранит:
- точку $x_{\text{node}}$, соответствующую медиане разбиения;
- номер оси $j$, по которой производилось разбиение;
- пороговое значение $m$;
- ссылки на левое и правое поддеревья.

Сложность построения KD‑дерева определяется стоимостью нахождения медианы на каждом уровне рекурсии. Точный линейный алгоритм поиска медианы (Blum et al., 1973) даёт общую сложность $O(n \log n)$. На практике часто применяют приближённые методы (медиана случайной подвыборки), что сохраняет среднюю сложность $O(n \log n)$, но лишает гарантий идеальной сбалансированности. Глубина дерева составляет $O(\log n)$ при равномерном распределении данных.

#### 4.1.2 Поиск ближайших соседей

Поиск одного ближайшего соседа (алгоритм легко обобщается на поиск $k$ ближайших путём поддержания очереди из $k$ лучших кандидатов) осуществляется рекурсивным обходом дерева. В начале поиска устанавливается начальное лучшее расстояние $\text{best\_dist} = +\infty$ и $\text{best\_point} = \text{None}$.

```
def search(node, query, best_dist, best_point):
    если node отсутствует:
        вернуть (best_dist, best_point)

    # Вычисляем расстояние от запроса до точки узла
    dist = d(query, node.point)
    если dist < best_dist:
        best_dist = dist
        best_point = node.point

    # Определяем, какое поддерево является ближайшим
    axis = node.axis
    diff = query[axis] - node.point[axis]

    если diff <= 0:
        near_child = node.left
        far_child  = node.right
    иначе:
        near_child = node.right
        far_child  = node.left

    # Рекурсивно обходим ближайшее поддерево
    (best_dist, best_point) = search(near_child, query, best_dist, best_point)

    # Проверка необходимости обхода дальнего поддерева
    если |diff| < best_dist:
        (best_dist, best_point) = search(far_child, query, best_dist, best_point)

    вернуть (best_dist, best_point)
```

Ключевая идея, обеспечивающая эффективность, — **правило отсечения**. После возврата из ближайшего поддерева мы имеем текущее наилучшее расстояние $\text{best\_dist}$. Расстояние от запроса до разделяющей гиперплоскости равно $|\text{diff}|$. Если $|\text{diff}| \ge \text{best\_dist}$, то все точки в дальнем поддереве находятся заведомо не ближе, чем текущий лучший кандидат, поскольку для любой точки $x$ из этого поддерева справедливо:

$$
d(\text{query}, x) \ge |\text{diff}| \ge \text{best\_dist}.
$$

В этом случае дальнее поддерево полностью исключается из рассмотрения, что и даёт выигрыш по сравнению с полным перебором.

**Сложность поиска.** При равномерном распределении данных и невысокой размерности $D$ среднее число посещаемых узлов составляет $O(\log n)$. Формально, анализ Фридмана, Бентли и Финкеля (Friedman, Bentley, Finkel, 1977) показывает, что время поиска $k$ ближайших соседей в KD‑дереве составляет $O(k \log n)$ в среднем для фиксированной размерности.

Однако при росте $D$ эффективность катастрофически падает. Это связано с тем, что объём $D$-мерного шара, определяющего окрестность запроса, экспоненциально мал по сравнению с объёмом ограничивающего его гиперкуба (ячейки Вороного, порождаемой разбиениями). В результате правило отсечения перестаёт срабатывать: почти все узлы дерева приходится посещать. Если $D$ сравнимо с $\log_2 n$ или превышает его, KD‑дерево вырождается до почти полного перебора, и сложность приближается к $O(Dn)$. На практике KD‑деревья эффективны при $D \lesssim 20$ и объёме выборки $n \gg 2^D$.

### 4.2 Точные методы: Ball‑дерево

Альтернативой KD‑дереву служит **Ball‑дерево** (Omohundro, 1989), которое разбивает пространство не гиперплоскостями, а вложенными гиперсферами. Каждый узел хранит центр $c \in \mathbb{R}^D$ и радиус $r \ge 0$ минимальной охватывающей сферы, содержащей все точки соответствующего поддерева. Листовые узлы содержат непосредственно точки данных.

Построение Ball‑дерева обычно осуществляется путём рекурсивного кластерного разбиения: выбирается центр, относительно него точки делятся на два кластера (например, методом 2‑средних или простым разбиением по наибольшей разности координат), и для каждого кластера строится минимальная охватывающая сфера.

Правило отсечения при поиске в Ball‑дереве использует неравенство треугольника. Если для текущего узла выполнено

$$
d(\text{query}, c) - r \ge \text{best\_dist},
$$

то минимально возможное расстояние от запроса до любой точки в поддереве (достигаемое на границе сферы в направлении запроса) уже превышает текущее наилучшее расстояние. Следовательно, всё поддерево может быть отброшено. Это условие часто оказывается более эффективным, чем прямоугольные отсечения KD‑дерева, особенно когда данные не вытянуты вдоль координатных осей. Ball‑деревья, как правило, сохраняют приемлемую производительность до $D \lesssim 30$.

### 4.3 Приближённые методы поиска

В большинстве крупномасштабных приложений (рекомендательные системы, поиск изображений, информационный поиск) нахождение строго ближайших соседей не является обязательным требованием. Достаточно, чтобы найденные точки были «достаточно близки» к запросу — например, попадали в $1+\varepsilon$ раз дальше истинного ближайшего соседа. Это наблюдение привело к развитию **приближённых методов** (approximate nearest neighbor search, ANNS), которые радикально ускоряют поиск, часто до сублинейного времени, ценой небольшой и контролируемой потери точности. Ниже кратко охарактеризованы три основные парадигмы, широко используемые в индустрии.

#### 4.3.1 Случайные проекции и Annoy

Алгоритм **Annoy** (Approximate Nearest Neighbors Oh Yeah), разработанный Эриком Бернхардссоном и до 2023 года применявшийся в Spotify для музыкальных рекомендаций, основан на построении леса бинарных деревьев, в каждом из которых пространство рекурсивно разделяется случайными гиперплоскостями. Для каждого дерева случайным образом выбираются две точки из обучающей выборки, и строится гиперплоскость, равноудалённая от них. Процесс повторяется рекурсивно до тех пор, пока в каждом листе не окажется не более $M$ точек (гиперпараметр). Запрос спускается по каждому дереву до соответствующего листа; объединение точек из всех посещённых листьев формирует множество кандидатов, среди которых уже можно выполнить точный поиск. Точность регулируется числом деревьев: больше деревьев — выше точность, но больше время поиска. Annoy прост в реализации, требует умеренной памяти, но плохо поддерживает онлайн-обновление данных.

#### 4.3.2 Локально-чувствительное хеширование (LSH)

**Локально-чувствительное хеширование** (Locality-Sensitive Hashing, LSH; Indyk, Motwani, 1998) основано на идее построения таких хеш-функций, что вероятность коллизии (попадания в один бакет) велика для близких точек и мала для далёких. Формально, семейство хеш-функций $\mathcal{H}$ называется $(R, cR, p_1, p_2)$-чувствительным, если для любых $x, x'$:

- если $d(x, x') \le R$, то $\Pr_{h \in \mathcal{H}}[h(x) = h(x')] \ge p_1$;
- если $d(x, x') \ge cR$, то $\Pr_{h \in \mathcal{H}}[h(x) = h(x')] \le p_2$,

где $c > 1$, $p_1 > p_2$. Для евклидова расстояния стандартное семейство строится на основе случайных проекций с квантованием:

$$
h_{w,b}(x) = \left\lfloor \frac{\langle w, x \rangle + b}{r} \right\rfloor,
$$

где $w \sim \mathcal{N}(0, I_D)$, $b \sim \text{Uniform}(0, r)$, а $r$ — ширина квантования. Для повышения контрастности между $p_1$ и $p_2$ применяют композицию нескольких хеш-функций в одну (AND-конструкция) и/или используют несколько независимых хеш-таблиц (OR-конструкция). На LSH основана библиотека **FAISS** (Facebook AI Similarity Search), широко применяемая для поиска по векторным представлениям.

#### 4.3.3 Иерархический навигационный мир малого радиуса (HNSW)

**HNSW** (Hierarchical Navigable Small World; Malkov, Yashunin, 2018) — графовый метод, на момент написания считающийся state-of-the-art для многих задач поиска ближайших соседей. Идея состоит в построении иерархии графов: на верхних слоях находятся относительно мало вершин, но рёбра между ними длинные (что обеспечивает быстрый переход в нужную область пространства), а на нижнем (нулевом) слое граф густой и обеспечивает точный локальный поиск. Каждый следующий слой строится вероятностным прореживанием предыдущего. Поиск начинается с произвольной вершины верхнего слоя, жадным спуском находится ближайшая вершина, затем процедура повторяется на следующем слое, и так до нулевого. HNSW обеспечивает логарифмическое время поиска и рекордные показатели точности, но предъявляет высокие требования к памяти. Алгоритм реализован в библиотеках `hnswlib` и `FAISS`.



Выбор структуры данных для поиска ближайших соседей определяется конкретными условиями задачи. При умеренных размерностях ($D \lesssim 20$) и объёмах выборки до $10^5$ KD‑деревья и Ball‑деревья предоставляют точный и прозрачный инструмент. В высокоразмерных пространствах ($D \gtrsim 50$) или при больших объёмах данных ($n \gtrsim 10^6$) предпочтение следует отдавать приближённым методам — Annoy, LSH или HNSW — которые дают контролируемый компромисс между точностью и скоростью. В современных библиотеках (`scikit-learn`, `FAISS`, `hnswlib`, `Annoy`) все перечисленные структуры данных реализованы и доступны для непосредственного использования.





## 5. Реализация с нуля и сравнение с библиотечной версией

Приведём реализацию k‑NN классификатора с нуля на Python. Будем использовать евклидово расстояние, единичные или взвешенные голоса.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

class SimpleKNN:
    def __init__(self, n_neighbors=5, weights='uniform'):
        self.n_neighbors = n_neighbors
        self.weights = weights

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        # матрица расстояний (n_query x n_train)
        dist = np.linalg.norm(X[:, np.newaxis, :] - self.X_train[np.newaxis, :, :], axis=2)
        # индексы k ближайших
        idx = np.argsort(dist, axis=1)[:, :self.n_neighbors]
        k_labels = self.y_train[idx]
        if self.weights == 'uniform':
            # большинство голосов
            pred = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=k_labels)
        else:
            # взвешенные веса
            dist_k = np.take_along_axis(dist, idx, axis=1)
            weights = 1.0 / (dist_k + 1e-8)
            # для каждого класса суммируем веса
            classes = np.unique(self.y_train)
            weighted_votes = np.zeros((X.shape[0], len(classes)))
            for j, c in enumerate(classes):
                mask = (k_labels == c)
                weighted_votes[:, j] = np.sum(weights * mask, axis=1)
            pred = classes[np.argmax(weighted_votes, axis=1)]
        return pred


Проверим на реальном датасете breast_cancer и сравним с реализацией sklearn.


In [ ]:
# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Стандартизация
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Наша реализация
my_knn = SimpleKNN(n_neighbors=5)
my_knn.fit(X_train_sc, y_train)
my_pred = my_knn.predict(X_test_sc)
print("My KNN accuracy:", accuracy_score(y_test, my_pred))

# Sklearn
sk_knn = KNeighborsClassifier(n_neighbors=5)
sk_knn.fit(X_train_sc, y_train)
sk_pred = sk_knn.predict(X_test_sc)
print("Sklearn KNN accuracy:", accuracy_score(y_test, sk_pred))



Результаты должны совпадать (с точностью до возможного различия в весовых схемах). Теперь визуализируем границы решений на синтетических двумерных данных (moons) для разных $k$.



In [ ]:
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_moons, y_moons, test_size=0.3, random_state=42)

def plot_decision_boundary(model, X, y, ax, title):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='viridis')
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k, ax in zip([1, 5, 15], axes):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_m, y_train_m)
    plot_decision_boundary(knn, X_train_m, y_train_m, ax, f'k = {k}')
plt.tight_layout()
plt.show()



Графики покажут, что при $k=1$ граница сильно изрезана и присутствуют «островки» (высокая дисперсия), при $k=15$ граница становится более гладкой, но хуже подстраивается под структуру данных (смещение).



## 6. Эксперименты: влияние $k$, метрики и размерности

Продолжим исследование на breast_cancer. Построим кривую зависимости точности на обучении и тесте от $k$.



In [ ]:
train_acc, test_acc = [], []
ks = range(1, 31)
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    test_acc.append(accuracy_score(y_test, knn.predict(X_test_sc)))

plt.figure(figsize=(8,5))
plt.plot(ks, train_acc, label='Train')
plt.plot(ks, test_acc, label='Test')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.title('Влияние k на точность')
plt.show()



Типичная картина: на обучении точность максимальна при $k=1$ и снижается с ростом $k$; на тесте наблюдается пик в районе $k \approx 5-10$. Это наглядно демонстрирует компромисс смещения и дисперсии.

Сравним разные метрики:




In [ ]:
metrics = ['euclidean', 'manhattan', 'cosine']
for m in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=m)
    knn.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_sc))
    print(f"Metric {m}: accuracy = {acc:.4f}")




Для низкоразмерных данных с числовыми признаками евклидова и манхэттенская метрики дают схожие результаты; косинусная может работать хуже, если не применяется специальная нормировка.

Важность стандартизации легко увидеть, сравнив точность на нестандартизованных и стандартизованных данных:



In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn.predict(X_test))
knn.fit(X_train_sc, y_train)
acc_sc = accuracy_score(y_test, knn.predict(X_test_sc))
print(f"Без стандартизации: {acc_raw:.4f}, со стандартизацией: {acc_sc:.4f}")




Без стандартизации точность значительно падает, так как признаки с большим масштабом подавляют вклад остальных в расстояние.

Для демонстрации проклятия размерности сгенерируем синтетические данные с двумя гауссовскими классами в пространствах разной размерности:



In [ ]:
from sklearn.datasets import make_classification

dims = [2, 10, 50, 100, 200]
accs = []
for D in dims:
    X_s, y_s = make_classification(n_samples=500, n_features=D, n_informative=D, n_redundant=0,
                                   n_clusters_per_class=1, random_state=42)
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_s, y_s, test_size=0.3, random_state=42)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_s)
    X_test_s = scaler.transform(X_test_s)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_s, y_train_s)
    acc = accuracy_score(y_test_s, knn.predict(X_test_s))
    accs.append(acc)
    print(f"D={D}: accuracy={acc:.3f}")

plt.figure(figsize=(8,5))
plt.plot(dims, accs, 'o-')
plt.xlabel('Число признаков D')
plt.ylabel('Accuracy')
plt.title('Проклятие размерности для k‑NN')
plt.grid(alpha=0.3)
plt.show()


С ростом $D$ точность k‑NN монотонно падает: при $D=200$ она часто приближается к случайному угадыванию.




## 7. Связь с байесовским подходом и другими методами

### 7.1 Вывод k‑NN из оценки плотности

Метод k‑ближайших соседей допускает элегантную вероятностную интерпретацию, которая проясняет его связь с байесовским решающим правилом. Рассмотрим задачу классификации с $C$ классами. Пусть обучающая выборка содержит $n_c$ объектов класса $c$, так что $\sum_{c=1}^C n_c = n$. Эмпирическая вероятность класса $c$ равна $\hat{P}(Y=c) = n_c / n$.

Зафиксируем точку $x \in \mathbb{R}^D$ и рассмотрим шар (или, в общем случае, окрестность) объёма $V$, содержащий ровно $k$ ближайших соседей точки $x$. Пусть среди этих $k$ соседей ровно $k_c$ принадлежат классу $c$, причём $\sum_{c=1}^C k_c = k$. Тогда **непараметрическая оценка совместной плотности** класса $c$ и признака $x$ имеет вид

$$
\hat{p}(x, Y=c) = \frac{k_c}{n V}.
$$

Действительно, в объёме $V$ находится $k_c$ точек класса $c$, а полный объём выборки равен $n$. Аналогично, **безусловная оценка плотности** признака $x$ равна

$$
\hat{p}(x) = \frac{k}{n V}.
$$

По формуле Байеса апостериорная вероятность класса $c$ при условии $x$ выражается как отношение совместной плотности к безусловной:

$$
\hat{P}(Y = c \mid X = x) = \frac{\hat{p}(x, Y=c)}{\hat{p}(x)} = \frac{k_c / (n V)}{k / (n V)} = \frac{k_c}{k}. \tag{7.1}
$$

Таким образом, доля представителей класса $c$ среди $k$ ближайших соседей является естественной непараметрической оценкой апостериорной вероятности. Решающее правило k‑NN

$$
\hat{y}(x) = \arg\max_{c} \frac{k_c}{k}
$$

есть не что иное, как эмпирическая аппроксимация **байесовского решающего правила**

$$
y^*(x) = \arg\max_{c} P(Y = c \mid X = x),
$$

которое минимизирует вероятность ошибочной классификации. Этот вывод не требует никаких параметрических предположений о форме распределений и опирается исключительно на локальную структуру данных.

### 7.2 Асимптотическая состоятельность

Ключевой теоретический результат о состоятельности метода k‑NN был получен Стоуном (Stone, 1977). Приведём его в формулировке для регрессии; аналогичное утверждение справедливо и для классификации.

**Теорема (Stone, 1977).** Пусть пары $(X_i, Y_i)$ независимы и одинаково распределены, причём $\mathbb{E}|Y| < \infty$. Предположим, что последовательность чисел соседей $k = k_n$ выбрана так, что

$$
k_n \to \infty \quad \text{и} \quad \frac{k_n}{n} \to 0 \qquad \text{при} \qquad n \to \infty.
$$

Тогда для почти всех $x$ относительно распределения $X$ оценка k‑NN является состоятельной:

$$
\hat{m}_n(x) = \frac{1}{k_n} \sum_{i \in N_{k_n}(x)} Y_i \xrightarrow{P} m(x) = \mathbb{E}[Y \mid X = x].
$$

Для задачи классификации риск k‑NN сходится к байесовскому риску $R^* = \mathbb{P}(\hat{y}(X) \neq Y)$.

**Интуитивный смысл условий.** Требование $k_n \to \infty$ обеспечивает применимость закона больших чисел: с ростом числа соседей выборочное среднее сходится к условному математическому ожиданию. Требование $k_n / n \to 0$ гарантирует, что окрестность стягивается к точке $x$, так что усреднение остаётся локальным и смещение не нарастает.

**Скорость сходимости.** Несмотря на асимптотическую состоятельность, практическая применимость k‑NN в высоких размерностях ограничена катастрофически медленной сходимостью. Для достаточно гладкой функции регрессии $m(x)$ среднеквадратичная ошибка метода k‑NN убывает как

$$
\mathbb{E}\bigl[ (\hat{m}_n(x) - m(x))^2 \bigr] = O\bigl( n^{-2/(2+D)} \bigr). \tag{7.2}
$$

Эта скорость экспоненциально деградирует с ростом $D$: при $D=10$ для уменьшения ошибки вдвое требуется увеличить выборку примерно в $2^{6}=64$ раза, а при $D=100$ — более чем в $10^{12}$ раз. Данное обстоятельство является формальным выражением проклятия размерности, обсуждавшегося в разделе 3.

### 7.3 Связь с ядерными методами

Как было показано в разделе 2, метод k‑NN является частным случаем более широкого класса **ядерных непараметрических оценок**. Оценка Надарая–Ватсона (2.2) использует произвольное ядро $K$ и фиксированную ширину окна $h$, тогда как k‑NN соответствует прямоугольному ядру и адаптивной ширине $h = d(x, x_{(k)})$, определяемой локальной плотностью данных. Адаптивность ширины окна является одновременно и преимуществом (лучшая подстройка под неравномерную плотность), и недостатком: в областях малой плотности окрестность может стать неоправданно большой, что ведёт к значительному смещению.

Обратно, методы с фиксированной шириной окна страдают от противоположной проблемы: в областях высокой плотности в окно попадает избыточно много точек, что увеличивает вычислительные затраты, а в областях низкой плотности — недостаточно, что приводит к высокой дисперсии. Компромиссные подходы (например, k‑NN с ядерными весами) сочетают адаптивность k‑NN по числу соседей с гладкостью ядерных весов.

### 7.4 Сравнение с другими методами классификации

По сравнению с параметрическими методами (логистическая регрессия, линейный дискриминантный анализ) метод k‑NN не делает предположений о глобальной форме разделяющей поверхности. Это даёт ему высокую гибкость и способность восстанавливать границы сколь угодно сложной формы при достаточном объёме данных. Ценой этой гибкости является отсутствие явной модели: нельзя указать, какой признак наиболее важен, а сама граница существует лишь в виде набора точек и не может быть описана компактной формулой.

По сравнению с другими непараметрическими методами:

- **Деревья решений и случайный лес** также являются непараметрическими, но строят кусочно-постоянную аппроксимацию путём адаптивного разбиения пространства, что даёт лучшую интерпретируемость и меньшую чувствительность к масштабу признаков.
- **Метод опорных векторов с ядром** использует функцию близости (ядро), но строит глобальную решающую границу, максимизирующую зазор; он менее чувствителен к проклятию размерности, но требует решения задачи квадратичного программирования.
- **Гауссовские процессы** задают вероятностное распределение непосредственно на пространстве функций и дают естественную оценку неопределённости предсказания, что не свойственно k‑NN.

На практике k‑NN часто используют как **бенчмарк** — простой, легко реализуемый метод, с которым сравнивают более сложные алгоритмы. Если сложная модель не демонстрирует статистически значимого превосходства над k‑NN, её внедрение вряд ли оправдано. Кроме того, k‑NN остаётся востребованным в задачах, где важна прозрачность принятия решений и возможность сослаться на конкретные прецеденты (например, кредитный скоринг с объяснением «клиенту отказано, потому что его профиль близок к профилям трёх заёмщиков с дефолтом»).



Метод k‑ближайших соседей, несмотря на свою концептуальную простоту, имеет глубокие теоретические основания. Он непосредственно связан с байесовским решающим правилом через непараметрическую оценку плотности, является состоятельным при мягких условиях на рост $k$ и объём выборки, но страдает от катастрофической потери эффективности в высоких размерностях. Понимание этих связей не только проясняет внутреннюю логику метода, но и позволяет осознанно выбирать между k‑NN и альтернативными подходами в зависимости от свойств конкретной задачи.




## Вопросы для самопроверки

### Для начинающих
1. Как работает k‑NN для классификации и регрессии?
2. Что происходит с границей решений при $k=1$ и при $k=n$?
3. Какие метрики расстояния вы знаете и в каких ситуациях они применяются?
4. Почему перед применением k‑NN необходимо стандартизировать признаки?
5. В чём проявляется проклятие размерности для k‑NN?
6. Как выбрать число соседей $k$ на практике?

### Для профессионалов
1. Докажите, что k‑NN является состоятельной оценкой условного среднего при $k\to\infty$ и $k/n\to0$.
2. Почему KD‑дерево деградирует при большой размерности? Дайте формальное обоснование.
3. Выведите оптимальную весовую схему для регрессии, минимизирующую локальную среднеквадратичную ошибку.
4. Сравните k‑NN с ядерной оценкой Надарая–Ватсона: в чём сходство и различие?
5. Как k‑NN можно интерпретировать как оценку отношения плотностей для классификации?

## Задания повышенной сложности
1. Реализуйте KD‑дерево для поиска k ближайших соседей и сравните время работы с наивным перебором при различных $n$ и $D$.
2. Докажите, что среднее евклидово расстояние между двумя равномерно распределёнными точками в $D$-мерном единичном гиперкубе асимптотически равно $\sqrt{D/6}$.
3. Постройте кривые обучения для k‑NN и логистической регрессии на breast_cancer; проанализируйте, как размер выборки влияет на смещение и дисперсию каждого метода.
4. Реализуйте взвешенную k‑NN регрессию и сравните её с обычной на данных с неравномерной плотностью признаков.
5. Докажите, что в пределе $n\to\infty, k\to\infty, k/n\to0$ байесовская ошибка классификации для k‑NN достигается (риск сходится к байесовскому риску).